# Vision Frontend Demo

This notebook demonstrates the representation-only pipeline in `vision/`:

1. Generate synthetic colored-square stimuli
2. Preview generated images
3. Extract layer activations from a timm model
4. Preprocess features (z-score + L2 norm)
5. Save and inspect cached outputs

No DAM integration is performed here.

In [2]:
from pathlib import Path
import sys
import json
import numpy as np
from PIL import Image

# Make imports work whether notebook is launched from repo root or vision/.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "vision").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from vision.image_generator import (
    build_preview_grid,
    generate_square_stimuli,
    metadata_to_dicts,
)
from vision.model_wrapper import VisionEmbeddingWrapper
from vision.feature_preprocess import LayerwisePreprocessor

ModuleNotFoundError: No module named 'torch'

In [ ]:
# Config
MODEL_NAME = "vit_base_patch16_224"
USE_PRETRAINED = False  # Set True if weights download is available
DEVICE = "cpu"
N_PER_COLOR = 3
BATCH_SIZE = 8
SEED = 0
OUTPUT_DIR = Path("results/vision")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

images, metadata = generate_square_stimuli(
    n_per_color=N_PER_COLOR,
    image_size=224,
    square_size=56,
    fixed_position=True,
    seed=SEED,
)

print(f"Generated {len(images)} images")
metadata[:2]

In [ ]:
preview = build_preview_grid(images[:18], ncols=6)
preview_path = OUTPUT_DIR / f"preview_notebook_{MODEL_NAME}.png"
preview.save(preview_path)
preview

In [ ]:
wrapper = VisionEmbeddingWrapper(
    model_name=MODEL_NAME,
    pretrained=USE_PRETRAINED,
    device=DEVICE,
    pooling="auto",
)

features_raw = wrapper.extract(images, batch_size=BATCH_SIZE)
preprocessor = LayerwisePreprocessor(use_zscore=True, l2_normalize=True)
features = preprocessor.fit_transform(features_raw)

{k: v.shape for k, v in features.items()}

In [ ]:
npz_path = OUTPUT_DIR / f"features_notebook_{MODEL_NAME}.npz"
payload = {}
for layer_name, feats in features.items():
    payload[f"features_{layer_name}"] = feats
payload["stimulus_id"] = np.array([m.stimulus_id for m in metadata], dtype=object)
payload["color_name"] = np.array([m.color_name for m in metadata], dtype=object)
payload["x"] = np.array([m.x for m in metadata], dtype=np.int32)
payload["y"] = np.array([m.y for m in metadata], dtype=np.int32)
np.savez_compressed(npz_path, **payload)

metadata_json_path = OUTPUT_DIR / f"metadata_notebook_{MODEL_NAME}.json"
metadata_json_path.write_text(json.dumps(metadata_to_dicts(metadata), indent=2), encoding="utf-8")

print(f"Saved: {npz_path}")
print(f"Saved: {metadata_json_path}")
print(f"Saved: {preview_path}")